In [16]:
# %pip install python-dotenv
# %uv add dspy

In [17]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))


### check aicodetools library

In [18]:
import time
import threading
import tiktoken
from collections import deque
import dspy
from dspy.utils.callback import BaseCallback


class SlidingWindowLimiter:
    """Rate limiter that enforces both request and token limits per rolling minute."""

    _instance = None
    _lock = threading.Lock()

    def __new__(cls, max_requests_per_min=1000, max_tokens_per_min=2_000_000):
        if cls._instance is None:
            cls._instance = super().__new__(cls)
            cls._instance.max_requests = max_requests_per_min
            cls._instance.max_tokens = max_tokens_per_min
            cls._instance.requests = deque()  # [(timestamp, tokens)]
            cls._instance._lock = threading.Lock()
            cls._instance.encoder = tiktoken.get_encoding("cl100k_base")
        return cls._instance

    def _cleanup(self, now):
        """Remove entries older than 60s."""
        while self.requests and now - self.requests[0][0] > 60:
            self.requests.popleft()

    def _count(self):
        """Total requests & tokens in current 60s window."""
        total_tokens = sum(t for _, t in self.requests)
        return len(self.requests), total_tokens

    def acquire(self, tokens_used=0):
        """Wait until request fits in sliding 60s window."""
        with self._lock:
            while True:
                now = time.time()
                self._cleanup(now)
                req_count, token_count = self._count()

                # Can fit in current 60s window?
                if (req_count < self.max_requests and
                        token_count + tokens_used <= self.max_tokens):
                    # Record the new request
                    self.requests.append((now, tokens_used))
                    break  # proceed

                # Otherwise, figure out when we can retry
                oldest_time = self.requests[0][0]
                sleep_time = max(0.01, 60 - (now - oldest_time))
                print(f"⚠️ Throttling: sleeping {sleep_time:.2f}s (req={req_count}, tokens={token_count})")
                time.sleep(sleep_time)


class DelayAndLogCallback(BaseCallback):
    """DSPy callback using sliding window limiter."""

    def __init__(self):
        self.limiter = SlidingWindowLimiter()

    def _estimate_tokens(self, messages=None, prompt=None):
        """Estimate token usage using tiktoken."""
        text = ""
        if prompt:
            text = str(prompt)
        elif messages:
            # concatenate all message contents
            text = " ".join(m.get("content", "") for m in messages)
        return len(self.limiter.encoder.encode(text))

    def on_lm_start(self, *args, **kwargs):
        inputs = kwargs.get("inputs") or {}
        prompt = inputs.get("prompt")
        messages = inputs.get("messages")
        tokens_used = self._estimate_tokens(messages=messages, prompt=prompt)
        self.limiter.acquire(tokens_used=tokens_used)

    def on_lm_end(self, *args, **kwargs):
        pass


In [19]:

# from aicodetools import ClientManager 

# code_tool_manager = ClientManager(
#                 "super-bench:latest", base_log_dir="runs/super/"
#             )

# code_tool_client = code_tool_manager.get_client('initial')

In [20]:
import os
# os.environ['OPENAI_API_KEY'/] = input()
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

True

In [21]:
import dspy
lm = dspy.LM("azure/gpt-4.1", temperature=1.0, num_retries=30, callbacks=[DelayAndLogCallback()])
tlm = dspy.LM("azure/gpt-4.1",temperature=1.0)
dspy.configure(lm=lm)

In [22]:
# print(lm("Say this is a test!") ) # => ['This is a test!']
print(lm(messages=[{"role": "user", "content": "Say this is a test!"}]))  # => ['This is a test!']
print(tlm(messages=[{"role": "user", "content": "t : Say this is a test!"}]))  # => ['This is a test!']

['This is a test!']
['This is a test!']


## Load the benchmark and view one example from the benchmark

In [23]:

from gepa_artifact.benchmarks.super_bench.super_utils import FinishResponse
from gepa_artifact.benchmarks.super_bench import benchmark as sb_metas

In [24]:
bench = sb_metas[0].benchmark()

In [25]:
len(bench.train_set), len(bench.val_set), len(bench.test_set)

(9, 9, 27)

In [26]:
import pprint
pprint.pprint(bench.train_set[0])

Example({'instance_id': 'pie-perf', 'github_repo': 'https://github.com/madaan/pie-perf', 'git_commit': 'ee1989b66756470622e3b89c4aa031f083f57ef9', 'query': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0). Once evaluated, report the result problem_id and input_acc for each problem of the dataset, as a json list of dictionaries structured as follows: [{"problem_id": "", "input_acc": 0.0}] (replace "" and 0.0 with the actual values).\n\nAdditional instructions:\n1. Set "num_trials": 2 in the evaluation configuration file to reduce computation time.\n2. Load only the first 10 rows of the dataset.\n\nGit repository: https://github.com/madaan/pie-perf', 'query_components': {'e2e_task': 'Evaluate the generations of my code improving model which are provided in https://drive.google.com/file/d/1izs1iF5cd_NAZsOaZvrrQF3NAsoP8lHf/view?usp=sharing (v1 vs v0).', 'scenario_task': '

## Load the program and display the program
The program is a 3-module system, each of which handles the urgency, sentiment and categories classification respectively

In [27]:
program = sb_metas[0].program[0]
program

react.react = Predict(StringSignature(query, github_repo, git_commit, trajectory -> next_thought, next_tool_name, next_tool_args
    instructions='Solve the question and provide the answer in the correct format.\n\nYou are an Agent. In each episode, you will be given the fields `query`, `github_repo`, `git_commit` as input. And you can see your past trajectory so far.\nYour goal is to use one or more of the supplied tools to collect any necessary information for producing `result`.\n\nTo do this, you will interleave next_thought, next_tool_name, and next_tool_args in each turn, and also when finishing the task.\nAfter each tool call, you receive a resulting observation, which gets appended to your trajectory.\n\nWhen writing next_thought, you may reason about the current situation and plan for future steps.\nWhen selecting the next_tool_name and its next_tool_args, the tool must be one of:\n\n(1) read_file, whose description is <desc>          Read file with optional line range and/or 

### Make Sure docker is installed and running

## Define an evaluator and evaluate the base program

In [28]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='super_react.json',
    save_as_csv='super_react.csv'
)

In [ ]:
evaluate(program)

## Load the GEPA Optimizer

In [37]:
# Import GEPA and define the optimizer
from gepa_artifact.gepa.gepa import GEPA,GEPAState
from gepa_artifact.utils.capture_stream_logger import Logger

import time

runs_dir = os.path.join(os.getcwd(), "runs", time.strftime("%Y-%m-%d_%H-%M-%S"))
os.makedirs(runs_dir, exist_ok=True)

gepa_logger = Logger(os.path.join(runs_dir, "run_log.txt"))

if sb_metas[0].feedback_fn_maps is None or sb_metas[0].feedback_fn_maps[0] is None:
    def feedback_func(predictor_output, predictor_inputs, module_inputs, module_outputs, captured_trace):
        pred = sb_metas[0].metric_with_feedback(module_inputs, module_outputs, None)
        return {
            "feedback_score": pred.score,
            "feedback_text": pred.feedback,
        }

    feedback_fn_map = {k:feedback_func for k, v in program.named_predictors()}
else:
    feedback_fn_map = sb_metas[0].feedback_fn_maps[0]

optimizer = GEPA(
    named_predictor_to_feedback_fn_map=feedback_fn_map,
    knowledgebase_qe=None,
    metric=sb_metas[0].metric,
    run_linearized_gepa=False,
    use_merge=True, 
    teacher_lm = tlm,
    set_for_merge_minibatch='val', 
    track_scores_on='val',
    num_iters=9,
    run_dir=runs_dir,
    logger=gepa_logger,
    num_threads=9)

## Optimize the program with GEPA

In [38]:
sb_metas[0].program[0].get_lm()

## RUN to optimise the program

In [39]:
# optimized_program = optimizer.compile(
#     sb_metas[0].program[0],
#     trainset=bench.train_set,
#     valset=bench.val_set,
# )

In [40]:
# optimizer.gepa_state.save(runs_dir)

## Load from the Saved dir

In [42]:
state = GEPAState.load('runs/2025-10-20_03-45-26')

In [43]:
def idxmax(lst):
    """Return the index of the maximum value in a list."""
    max_val = max(lst)
    return lst.index(max_val)

In [44]:
gepa_state = state
best_prog_idx = idxmax(gepa_state.per_program_tracked_scores)
best_prog = gepa_state.program_candidates[best_prog_idx]

In [45]:
optimized_program=best_prog

### Let's print the prompts that GEPA discovered

In [48]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

Predictor: react.react
Prompt:
You are an advanced ML experiment execution assistant focused on running controlled, reproducible machine learning benchmarks from academic repositories containing code, datasets, and configuration scripts.

Your tasks follow this recurring structure:
- You receive as input:
  - A `query` describing a controlled ML run with precise instructions on experiment, dataset restrictions, hyperparameter changes, and the needed format/result metric extraction (often a rubric-specified JSON dictionary).
  - The `github_repo` URL and `git_commit` specifying the codebase/commit to use.
  - Sometimes a `trajectory`—log of previous steps/actions/results for multi-step tasks.

Here are critical, domain-specific and general instructions for performing these experiments reliably and as intended:

**Repository and Directory Navigation**
- All tasks assume the codebase has been cloned at the specified commit; however, you must always verify correct repo state before attempt

## Now, let's evaluate the optimized program

In [49]:
import dspy
evaluate = dspy.Evaluate(
    devset=bench.test_set,
    metric=sb_metas[0].metric,
    num_threads=9,
    display_table=True,
    display_progress=True,
    max_errors=100 * len(bench.test_set),
    provide_traceback = True,
    failure_score=0,
    save_as_json='optimized_react.json',
    save_as_csv='optimized_react.csv'
)

In [23]:
evaluate(optimized_program)

  0%|          | 0/27 [00:00<?, ?it/s]Available Tools for g-transformer: on runtime aicodetools-g-transformer-018464ad 4
Available Tools for spa: on runtime aicodetools-spa-1c293ae3 4
Available Tools for mezo: on runtime aicodetools-mezo-723a0405 4
Available Tools for mode-connectivity-plm: on runtime aicodetools-mode-connectivity-plm-770f1895 4
Available Tools for mbib: on runtime aicodetools-mbib-c5fe66d3 4
Available Tools for unsupervisedhierarchicalsymbolicregression: on runtime aicodetools-unsupervisedhierarchicalsymbolicregression-0daa9628 4
Available Tools for conv_graph: on runtime aicodetools-conv_graph-8e5204e7 4
Available Tools for pira: on runtime aicodetools-pira-e53a9588 4
Available Tools for pet: on runtime aicodetools-pet-ecd618dd 4
Cleaned up Tools g-transformer : True  True
success=False structured_output={'Sentence-level BLEU': 0.0, 'Document-level BLEU': 0.0} reasoning="The specified repository (https://github.com/baoguangsheng/g-transformer) is absent in the workin

2025/10/20 08:29:49 WARNING dspy.predict.react: Ending the trajectory: Agent failed to select a valid tool: 
Traceback (most recent call last):
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/adapters/chat_adapter.py", line 169, in parse
    fields[k] = parse_value(v, signature.output_fields[k].annotation)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/gepa_artifact/utils/dspy/dspy/adapters/utils.py", line 173, in parse_value
    return TypeAdapter(annotation).validate_python(candidate)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/mnt/c/Users/2825425/work/gepa-research/gepa-modified/.venv/lib/python3.12/site-packages/pydantic/type_adapter.py", line 441, in validate_python
    return self.validator.validate_python(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
pydantic_core._pydantic_core.ValidationError: 1 validation error for dict[str,

Cleaned up Tools mezo : True  True
success=False structured_output={'accuracy': 0.0, 'dev_accuracy': 0.0} reasoning='Step 1 (Repository Preparation) failed: The workspace directory is empty, with no files or subdirectories. The MeZO repository (https://github.com/princeton-nlp/MeZO at commit 552cb1b710767f9a6e1dc8f9645d7640376f9941) was not cloned, so all subsequent protocol steps (code inspection, dataset loading, script execution) are blocked. As per protocol, no experiment can be run until the repository and required scripts are present. This output reflects early termination with placeholders, per strict instructions.' summary='Experiment was terminated at Step 1 because the MeZO repository was missing from the workspace. No files or scripts were available to inspect or execute. To proceed, the repository must be cloned at the specified commit.'
Stepwise protocol review:

Step 1: Repository Preparation  
- Checked the current working directory (/workspace) and listed contents with 

2025/10/20 08:41:27 INFO dspy.evaluate.evaluate: Average Metric: 2.458333333333333 / 27 (9.1%)


,instance_id,github_repo,git_commit,query,query_components,answer,landmarks,trajectory,reasoning,result,super_score
0,g-transformer,https://github.com/baoguangsheng/g-transformer,dcc7695ceb0ecc3250e1c28215e9ddcd22700b39,Use the https://github.com/baoguangsheng/g-transformer repository ...,{'e2e_task': 'Use the https://github.com/baoguangsheng/g-transform...,"{""Sentence-level BLEU"": 0.0, ""Document-level BLEU"": 0.01}",['INFO\\] Building segmented data' 'INFO \\| fairseq_cli.preproces...,"{'thought_0': ""Before configuring or running the experiment, I nee...",Stepwise reasoning: - Step 1 (Repository Preparation): Attempted t...,"success=False structured_output={'Sentence-level BLEU': 0.0, 'Docu...","✔️ [Prediction(\n score=0.25,\n score_dict={'submitted': 1, ..."
1,spa,https://github.com/OceannTwT/SPA,a8bb190a182c076f80d90ac59921abd1df29b0ae,"Train the SPA model on alpaca_data_en_52k (from the repo), startin...",{'e2e_task': 'Train the SPA model on alpaca_data_en_52k (from the ...,"{""training_loss"": 4.97342586517334}","[Generating train split: \d+ examples, >> \*\*\*\*\* Running train...","{'thought_0': 'First, I need to check the contents of the current ...",Stepwise Reasoning: 1. **Repository Preparation:** Upon beginning ...,success=False structured_output={'training_loss': 0.0} reasoning='...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
2,mezo,https://github.com/princeton-nlp/MeZO,552cb1b710767f9a6e1dc8f9645d7640376f9941,"Train using the ""MeZO"" method (no prefix-tuning / lora) on the RTE...","{'e2e_task': 'Train using the ""MeZO"" method (no prefix-tuning / lo...","{""accuracy"": 0.8, ""dev_accuracy"": 0.4}",['- INFO - Sample train set \\d+/\\d+' '- INFO - \\*\\*\\*\\*\\* R...,"{'thought_0': 'Before running the experiment, I need to verify tha...",Stepwise protocol review: Step 1: Repository Preparation - Checked...,"success=False structured_output={'accuracy': 0.0, 'dev_accuracy': ...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
3,mode-connectivity-plm,https://github.com/thunlp/mode-connectivity-plm,f45efd03fabdb23777c3ddcb35612cb1f7b1960c,"Fine-tune roberta-base on a single-task, ""rotten tomatoes"" which i...","{'e2e_task': 'Fine-tune roberta-base on a single-task, ""rotten tom...","{""eval_loss"": 0.0368}","[Running tokenizer on dataset:, \*\*\*\*\* train metrics \*\*\*\*\...","{'thought_0': ""Before proceeding with the experiment setup, I need...",Step-by-step reasoning for this experiment request: 1. **Repositor...,"success=False structured_output={'eval_loss': 0.0} reasoning=""The ...","✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
4,mbib,https://github.com/Media-Bias-Group/MBIB,b9a887ffd461fa462e89835fc27b36e370091954,"Train a bart-base model on the ""linguistic-bias"" task using the ba...","{'e2e_task': 'Train a bart-base model on the ""linguistic-bias"" tas...","{""average_weighted_f1"": 0.44272727272727275}","[Training Initialized for fold \d+, The current dev loss: tensor\(...",{'thought_0': 'I need to begin by inspecting the current directory...,"Step 1: Repository Verification Upon starting, I inspected the cur...",success=False structured_output={'average_weighted_f1': 0.0} reaso...,"✔️ [Prediction(\n score=0.0,\n score_dict={'submitted': 1, '..."
5,unsupervisedhierarchicalsymbolicregression,https://github.com/SiyuLou/UnsupervisedHierarchicalSymbolicRegression,9fb1ba66ee1006f14c15f1b5b2390c8d0fa24fd6,Train a hierarchical symbolic regression model from this repo usin...,{'e2e_task': 'Train a hierarchical symbolic regression model from ...,"{""Rf_final"": {""R_square"": -0.25985264778137207, ""MSE"": 0.148261207...","[the size of train dataset is \d+, validation_dataset is \d+, test...","{'thought_0': ""To begin, I need to verify that I'm in the correct ...",Step 1: Repository Preparation I began by verifying the presence o...,"success=False structured_output={'Rf_final': {'R_square': 0.0, 'MS...","✔️ [Prediction(\n score=0.08333333333333333,\n

9.1

Available Tools for team: on runtime aicodetools-team-0f6d0b45 4
Available Tools for cet: on runtime aicodetools-cet-eddc9a60 4
Available Tools for linkbert: on runtime aicodetools-linkbert-b1b54d9b 4
Cleaned up Tools team : True  True
success=False structured_output={'classification_acc': 0.0, 'classification_macro_f1': 0.0, 'instance_acc': 0.0} reasoning='The TEAM codebase at commit e43753fde8e53e498cf3056b85ae2f306902121f was not found in the working directory (/workspace) or its subdirectories. There are no scripts, folders, or files to run, inspect, or modify per task requirements. Therefore, the experiment protocol was halted before any dataset preparation, script inspection, or experiment configuration could occur. Returned placeholder metrics as required.' summary='Terminated at repository preparation step; the TEAM codebase was entirely missing, so no experiment setup or execution could occur. Placeholder metrics provided, success set to false.'
Step 1: Repository Preparation 